# Five-token autocomplete · GRPO lab (revision 2)

Run on a **fresh Colab GPU runtime**. This notebook embeds the source and CPU checkpoint; no uploads or ZIP are needed.

Revision 2 uses sparse action-token training and output-head LoRA to reduce GPU memory, pins Transformers/PEFT, streams complete subprocess errors, saves logs, and runs a short GPU training check before the full experiment.

All business data and value estimates are synthetic. CPU mechanics and small causal-LM integration were tested; the original failed GPU run cannot be diagnosed from exit code 1 alone. GPU speed and quality still need your run.

Setup removes the unused torchao package to resolve the confirmed Colab torchao 0.10 / PEFT incompatibility.

The results cell now displays a readable comparison and examples. Downloads contain only a small results report. SGLang model export is optional and disabled by default.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, io, zipfile, base64
ROOT = Path('/content/autocomplete-grpo-v2') if Path('/content').is_dir() else Path.cwd()/'autocomplete-grpo-v2'
ROOT.mkdir(parents=True, exist_ok=True)
PAYLOAD = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as z:
    for member in z.infolist():
        if not (ROOT/member.filename).resolve().is_relative_to(ROOT.resolve()):
            raise ValueError('Invalid archive path')
    z.extractall(ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Drop any v1 imports if the setup is rerun in an existing kernel.
for name in list(sys.modules):
    if name == 'autocomplete_grpo' or name.startswith('autocomplete_grpo.'):
        del sys.modules[name]
from autocomplete_grpo.runner import run_visible
run_visible([sys.executable, '-m', 'pip', 'install', '-e', '.'], ROOT, 'setup.log')
print('Revision 2 ready:', ROOT)


## 1 · Read the executed result

The shipped run used 250 GRPO steps and 200 held-out contexts. Value is an expected synthetic currency amount per context, including the ignore-all path. The greedy optimizer is the strong baseline; GRPO was approximately tied with it.


In [ ]:
from IPython.display import display, HTML
import html
report = json.loads((ROOT/'results/cpu/metrics.json').read_text())
rows = ''.join(f"<tr><td>{html.escape(name)}</td><td>{m['simulated_expected_gmv']:.3f}</td><td>{m['fallback_rate']:.1%}</td></tr>" for name,m in report['metrics'].items())
display(HTML('<table><tr><th>Method</th><th>Simulated value</th><th>Fallbacks</th></tr>'+rows+'</table>'))
print('GRPO minus greedy:', report['metrics']['grpo_policy']['delta_vs_greedy_list_value'])


## 2 · Inspect the five suggestions

Change the seed to create another shopper, category and candidate pool. This uses the trained CPU policy and performs real inference; it does not call a hosted service.


In [ ]:
import numpy as np
from autocomplete_grpo.data import generate
from autocomplete_grpo.cpu import decode
from autocomplete_grpo.reward import popularity, direct_value, greedy_value, deploy_slate, expected_value
weights = np.load(ROOT/'results/cpu/policy.npz')
def inspect_context(seed=87):
    row = generate(1, seed=seed)[0]
    print('Typed:', row['prefix'])
    print(row['session'])
    methods = {'Popularity': popularity(row), 'Direct value': direct_value(row),
               'Greedy list': greedy_value(row), 'GRPO': decode(row, weights['grpo'])}
    for name, raw in methods.items():
        slate, fallback = deploy_slate(row, raw)
        print(f"\n{name} | simulated value={expected_value(row, slate, oracle=True):.3f} | fallback={fallback}")
        for rank, i in enumerate(slate, 1):
            print(f"  {rank}. {row['candidates'][i]['query']}")
try:
    import ipywidgets as widgets
    widgets.interact(inspect_context, seed=widgets.IntSlider(value=87,min=1,max=200,continuous_update=False))
except ImportError:
    inspect_context(87)


## 3 · Reproduce CPU training

This takes roughly a minute or a few minutes depending on your CPU. The output folder is separate from the included result. The objective is clipped group-relative policy optimization plus exact categorical KL to the frozen supervised policy.


In [ ]:
run_visible([sys.executable, '-u', '-m', 'autocomplete_grpo.cpu', '--steps', '250', '--out', 'results/cpu-rerun'], ROOT, 'cpu-training.log')


## 4 · Small pretrained LLM on GPU

Choose Runtime → Change runtime type → GPU. Install the pinned packages, then run the preflight and a 3-step SFT / 2-step GRPO check. The full run starts only when those succeed.

The base embeddings are frozen. Only twenty input-token rows plus LoRA adapters are trained. No full embedding/head optimizer states are created.


In [ ]:
from autocomplete_grpo.runner import run_visible
run_visible([sys.executable, '-m', 'pip', 'install', '-e', '.[gpu]'], ROOT, 'install.log')
# Colab may preinstall torchao 0.10, which blocks PEFT's LoRA dispatcher.
# This non-quantized prototype does not use that optional package.
run_visible([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], ROOT, 'torchao-cleanup.log')
# A fresh subprocess reads the installed packages, avoiding stale notebook imports.
run_visible([sys.executable, '-u', '-m', 'autocomplete_grpo.runner'], ROOT, 'preflight.log')


In [ ]:
from autocomplete_grpo.runner import run_visible
run_visible([sys.executable, '-u', '-m', 'autocomplete_grpo.data', '--train', '1000'], ROOT, 'data.log')
# Fast compatibility check before investing in the full experiment.
run_visible([sys.executable, '-u', '-m', 'autocomplete_grpo.llm',
    '--model', 'Qwen/Qwen2.5-0.5B-Instruct', '--sft-steps', '3', '--steps', '2',
    '--group', '2', '--eval-n', '2', '--out', 'results/gpu-check'], ROOT, 'gpu-check.log')
# If this fails, the actual traceback is shown here and saved in results/logs/.
run_visible([sys.executable, '-u', '-m', 'autocomplete_grpo.llm',
    '--model', 'Qwen/Qwen2.5-0.5B-Instruct', '--sft-steps', '100', '--steps', '100',
    '--group', '2', '--eval-n', '30', '--out', 'results/llm'], ROOT, 'training.log')


In [ ]:
from autocomplete_grpo.report import show_results
report, small_bundle = show_results(ROOT)


## 5 · Optional: export a model for SGLang

Leave this disabled to read or download experiment results. A merged model is large and is only needed when you are ready to benchmark SGLang.


In [ ]:
EXPORT_FOR_SGLANG = False  # Change only when you need a merged serving model.
if EXPORT_FOR_SGLANG:
    run_visible([sys.executable, '-u', '-m', 'autocomplete_grpo.export',
        '--adapter', 'results/llm/grpo', '--out', 'results/merged'], ROOT, 'export.log')
else:
    print('Model export skipped. Your readable results are already available above.')


```bash
python -m sglang.launch_server --model-path results/merged --host 127.0.0.1 --port 30000 --enable-custom-logit-processor
# Another terminal in the same project:
python -m autocomplete_grpo.benchmark --model results/merged --requests 200 --concurrency 1 --out results/latency-c1.json
python -m autocomplete_grpo.benchmark --model results/merged --requests 500 --concurrency 32 --qps 100 --out results/latency-qps100.json
```

Record p50/p95/p99, QPS, input tokens, validity, and errors. Five output tokens do not eliminate prompt prefill or queueing. The client supports comparison against another configured server, but an EAGLE draft must be compatible with the added action vocabulary and sampling constraints. No speculative speedup is assumed.

## 6 · Save results from Colab


In [ ]:
from autocomplete_grpo.report import show_results
# Four small files: summary, metrics CSV, readable examples, and compact details.
# Model weights, checkpoints, merged models, and old ZIPs are excluded.
report, small_bundle = show_results(ROOT, download=True)


## Optional: download the trained adapter
Only enable this if you want to keep model weights. It is separate from the small results report.


In [ ]:
DOWNLOAD_ADAPTER = False
if DOWNLOAD_ADAPTER:
    import shutil
    adapter = ROOT/'results/llm/grpo'
    if not adapter.is_dir():
        raise FileNotFoundError('Complete LLM training before downloading the adapter.')
    bundle = shutil.make_archive(str(ROOT/'autocomplete-grpo-adapter'), 'zip', adapter)
    print(f'Adapter archive: {Path(bundle).stat().st_size/1024**2:.1f} MB')
    try:
        from google.colab import files
        files.download(bundle)
    except ImportError:
        print(bundle)
